# Existing stuff

### XGBModel

In [ ]:
from tqdm import tqdm
import numpy as np
import pandas as pd
from typing import Union, Optional, Dict, List, Literal, Tuple
import xgboost as xgb 
from src.models.base.basemodel import BaseModel
from matplotlib.axes import Axes

from src.utils.textformatting import align, section
from src.dataloading.deepdataloader import DeepDataLoader

import seaborn as sns

import matplotlib.pyplot as plt


from src.utils import traincolor, valcolor
from matplotlib.figure import Figure 

class EXISTINGSpatioTemporalXGBModel(BaseModel):

    """
    
    Examples:
    --------
    >>> dataloader_identity_graph = DeepDataLoader(disease_name, get_data_env(), nuts_level=nuts_level, min_date=min_date,max_date=max_date, include_population=False, horizon_size = horizon_size, horizon_leadtime = horizon_leadtime, sequence_length=sequence_length, split_berlin=split_berlin)
    >>> dataloader_identity_graph.add_time_features()
    >>> dataloader_identity_graph.log_transform_target()
    >>> dataloader_identity_graph.set_splits(split_trainval, split_valtest)
    >>> dataloader_identity_graph.normalize()
    >>> dataloader_identity_graph.add_lagged_features(lags = lags)
    >>> dataloader_identity_graph.finalize()
    >>> dataloader_identity_graph.retrieve_graph('identity_selfmax').construct_dataloaders()

    >>> xgbmodel_id = SpatioTemporalXGBModel(dataloader_identity_graph)
    >>> xgbmodel_id.set_global_hparams(lr = 0.01, n_epochs=500)
    >>> xgbmodel_id.set_model_hparams(n_estimators = 1000, max_depth = 8)
    >>> xgbmodel_id.train(verbose = 2)
    >>> xgbmodel_id.forecast('train')
    >>> xgbmodel_id.show_forecasts('train',26, timeframe = ['2009-01-01','2010-01-01'])    
    """


    def __init__(self, 
                 dataloader     : 'DeepDataLoader',
                 name           : Optional[str] = None):
        super().__init__(dataloader, name)
        
        if not self.name:
            self.name = f'SpatioTemporalXGB'
        
        # dataloader metadata
        self.gnn_dataloader     = dataloader
        self.sequence_length    = dataloader.sequence_length
        self.horizon_size       = dataloader.horizon_size
        self.models             = {}
        
        # Store graph structure if provided
        self.edge_index         = dataloader.edge_index
        self.neighbor_dict: Optional[Dict[int, list]] = None
        
        self._build_neighbor_dict()
        
        # XGBoost models (one per horizon if multi-horizon)
        
        # Data containers
        self.long_df_train = self._unpack_dataloader(dataloader.dataloader_train)
        self.long_df_val   = self._unpack_dataloader(dataloader.dataloader_val)
        self.long_df_test  = self._unpack_dataloader(dataloader.dataloader_test)    

        target_col = f"target_h{0}"
        feature_cols = [col for col in self.long_df_train.columns
                        if col not in ['t_idx', 'node','timestamp'] and not col.startswith('target')]
        # Prepare X and y as NumPy arrays (for sklearn API)
        self.X_train = self.long_df_train[feature_cols].values
        self.y_train = self.long_df_train[target_col].values

        self.X_val = self.long_df_val[feature_cols].values
        self.y_val = self.long_df_val[target_col].values

        self.X_test = self.long_df_test[feature_cols].values
        self.y_test = self.long_df_test[target_col].values

        self._map_timesteps()    
        
        self._state = {
            'model_initialized' : False,
            'trained'           : False,
            'forecasted'        : False,
        }
        
        self.evaluation_datasets    = {}
        self.train_losses           = []
        self.val_losses             = []
       
    def _check_state(self, required_states: list) -> None:
        """Validate that required setup steps have been completed."""
        missing = [s for s in required_states if not self._state.get(s, False)]
        if missing:
            raise ValueError(
                f"Missing required setup steps: {', '.join(missing)}. "
                f"Call the corresponding methods first."
            )
    
    def _build_neighbor_dict(self, taskbar = False):
        """
        Build dictionary mapping each node to its neighbors
        
        """
        
        neighbor_dict   = {}
        edge_list       = self.edge_index.t().tolist()
        
        iterator = tqdm(edge_list, desc='walking edge indices') if taskbar else edge_list

        for src, dst in iterator:
            if src not in neighbor_dict:
                neighbor_dict[src] = set()
            neighbor_dict[src].add(dst)
        
        
        self.neighbor_dict = {k: sorted(list(v)) for k, v in neighbor_dict.items()}
    
    def _unpack_dataloader(self, loader):
        """
        Extract data from GNN dataloader format into flat dataframe (vectorized),
        and augment with neighbor-aggregated features.
        """

        import numpy as np
        import pandas as pd
        from tqdm import tqdm

        num_nodes, num_features, sequence_length = loader[0].x.shape
        num_snapshots = len(loader)

        features = np.empty((num_snapshots, num_nodes, num_features, sequence_length), dtype=np.float32)
        targets = np.empty((num_snapshots, num_nodes, self.horizon_size), dtype=np.float32)

        for t_idx, snapshot in enumerate(loader):
            features[t_idx] = snapshot.x.numpy()
            targets[t_idx] = snapshot.y.numpy()

        # Reverse time axis to match naming convention (t, t-1, ...)
        features = features[..., ::-1]

        # Create feature column names
        feature_cols = []
        for feat_name in self.dataloader.feature_columns:
            for t in range(sequence_length):
                suffix = "_t" if t == 0 else f"_t-{t}"
                feature_cols.append(f"{feat_name}{suffix}")

        # Flatten arrays
        features_flat = features.reshape(num_snapshots * num_nodes, -1)
        targets_flat = targets.reshape(num_snapshots * num_nodes, self.horizon_size)

        # Create base dataframe
        t_idx_arr = np.repeat(np.arange(num_snapshots), num_nodes)
        node_arr = np.tile(np.arange(num_nodes), num_snapshots)

        df = pd.DataFrame(features_flat, columns=feature_cols)
        df['t_idx'] = t_idx_arr
        df['node'] = node_arr

        # Add target columns
        for h in range(self.horizon_size):
            df[f"target_h{h}"] = targets_flat[:, h]

        # === Vectorized Neighbor Aggregation ===
        # 1. Create neighbor mapping table — only valid neighbors
        rows = []
        for node, neighbors in self.neighbor_dict.items():
            for nbr in neighbors:
                if nbr != node:  # Optional: skip self if needed
                    rows.append({'node': node, 'neighbor': nbr})
        neighbor_map = pd.DataFrame(rows)  # [node, neighbor]

        # ⛔️ If the above DataFrame is empty (e.g. only self-loops), stop early
        if neighbor_map.empty:
            print("⚠️ No neighbors found — skipping aggregation.")
            return df

        # 2. Get node-level features only (no targets, no duplicate columns)
        base_feats = df.drop(columns=[c for c in df.columns if c.startswith('target')])

        # 3. Merge to assign neighbors
        df_neighbors = neighbor_map.merge(base_feats, left_on='neighbor', right_on='node')
        df_neighbors = df_neighbors.rename(columns={'node_x': 'node', 't_idx': 't_idx', 'node_y': 'neighbor'})

        # 4. Aggregate neighbor features
        agg_feature_cols = [col for col in base_feats.columns if col not in ['node', 't_idx']]

        neighbor_agg = (
            df_neighbors
            .groupby(['node', 't_idx'])[agg_feature_cols]
            .mean()
            .add_suffix('_neighbor')
            .reset_index()
        )

        # 5. Merge neighbor features back into original
        df_final = df.merge(neighbor_agg, on=['node', 't_idx'], how='left')

        return df_final
    
    def _map_timesteps(self):
        max_t = max(
            self.long_df_train["t_idx"].max(),
            self.long_df_val["t_idx"].max(),
            self.long_df_test["t_idx"].max()
        )

        # Create a mapping from t_idx to actual timestamp
        t_idx_to_ts = pd.date_range(start=self.dataloader.min_date, periods=max_t + 1, freq='W-MON')
        t_idx_to_ts = pd.Series(t_idx_to_ts, name="timestamp").reset_index().rename(columns={"index": "t_idx"})

        # Merge into each dataset
        for attr in ["long_df_train", "long_df_val", "long_df_test"]:
            df = getattr(self, attr)
            df = df.merge(t_idx_to_ts, on="t_idx", how="left")
            setattr(self, attr, df)        

    def forecast(self, 
                dataset: Literal['train', 'val', 'test'] = 'test'):
        """
        Generate forecasts for the specified dataset using trained XGBoost models.
        
        Parameters:
        -----------
        dataset : str
            Which dataset to forecast on ('train', 'val', 'test')

        Returns:
        --------
        self : SpatioTemporalXGBModel
        """
        horizon = 0
        model = self.models[f'horizon_{horizon}']
        
        # Select the appropriate dataframe
        X_map = {
            'train' : self.X_train,
            'val'   : self.X_val,
            'test'  : self.X_test
        }

        y_map = {
            'train' : self.y_train,
            'val'   : self.y_val,
            'test'  : self.y_test
        }

        eval_df_map = {
            'train' :  self.long_df_train[['timestamp','node','target_h0']],
            'val'   :  self.long_df_val[['timestamp'  ,'node','target_h0']],
            'test'  :  self.long_df_test[['timestamp' ,'node','target_h0']]                       
        }

        # Generate predictions per horizon
        preds = model.predict(X_map[dataset])

        horizon_prediction_dict = {}

        evaluation_df           = eval_df_map[dataset].rename(columns = {'target_h0': 'incidence'})
        evaluation_df['pred']   = preds

        horizon_prediction_dict['transformed']      = {f'horizon_{horizon}': evaluation_df}
        horizon_prediction_dict['nontransformed']   = {f'horizon_{horizon}': self._denorm_predictions(evaluation_df)}    
        self.evaluation_datasets[dataset] = horizon_prediction_dict    
        self._state['forecasted'] = True

        mse =  np.mean((evaluation_df['incidence'] - evaluation_df['pred']) ** 2)

        print(f'{dataset} mse: {mse:.4f}')

    def set_model_hparams(self,
                        max_depth: int = 6,
                        subsample: float = 0.8,
                        colsample_bytree: float = 0.8,
                        reg_alpha: float = 0.0,
                        reg_lambda: float = 1.0,
                        **kwargs):
        """
        Set XGBoost-specific hyperparameters (model-level).
        Excludes training-related hyperparams like learning_rate, n_estimators, etc.
        """
        self._check_state(['global_hparams_set'])

        # Protect global hparams from being overwritten here
        forbidden_keys = {'learning_rate', 'n_estimators', 'early_stopping_rounds'}
        filtered_kwargs = {k: v for k, v in kwargs.items() if k not in forbidden_keys}

        self.model_hparams = {
            'max_depth': max_depth,
            'subsample': subsample,
            'colsample_bytree': colsample_bytree,
            'reg_alpha': reg_alpha,
            'reg_lambda': reg_lambda,
            'tree_method': 'hist',
            'random_state': 42,
            **filtered_kwargs
        }

        self.config_info['model_hparams'] = self.model_hparams
        self._state['model_initialized'] = True

        # Initialize one model per horizon (with training params added later)
        self.models: Dict[str, xgb.XGBRegressor] = {}
        for hh in range(self.horizon_size):
            self.models[f'horizon_{hh}'] = xgb.XGBRegressor(**self.model_hparams)

        print(f"✓ Initialized {self.horizon_size} XGBoost models")

    def set_global_hparams(self, 
                        lr: float = 0.001,
                        n_epochs: int = 5,
                        patience: int = 15,
                        min_delta: float = 1e-4,
                        loss: Literal['rmse'] = 'rmse'):
        """
        Set global training hyperparameters (used across models).
        """
        
        if loss != 'rmse':
            raise ValueError('eval metrics other than rmse are not allowed')

        self.global_hparams = {
            'lr': lr,
            'n_epochs': n_epochs,
            'patience': patience,
            'min_delta': min_delta
        }

        self.lr = lr
        self.n_epochs = n_epochs
        self.patience = patience
        self.min_delta = min_delta
        self.loss    = loss

        self.config_info['global_hparams'] = self.global_hparams
        self._state['global_hparams_set'] = True

        # import xgboost as xgb

    def train(self, horizon=0, verbose=2, show_loss: bool = True):
        """
        Train XGBoost model for a given prediction horizon using global training hyperparameters.
        """
        self._check_state(['model_initialized', 'global_hparams_set'])

        model = self.models[f'horizon_{horizon}']

        # Inject global training hparams into the model
        model.set_params(
            eval_metric=self.loss,
            learning_rate=self.lr,
            n_estimators=self.n_epochs,
            early_stopping_rounds=self.patience
        )

        model.fit(
            self.X_train,
            self.y_train,
            eval_set=[(self.X_train, self.y_train), (self.X_val, self.y_val)],
            verbose=verbose > 1,
            
        )

        # Store losses
        train_pred = model.predict(self.X_train)
        val_pred = model.predict(self.X_val)

        train_mse = np.mean((train_pred - self.y_train) ** 2)
        val_mse = np.mean((val_pred - self.y_val) ** 2)

        if verbose > 0:
            print(f"✓ Horizon {horizon} - Train MSE: {train_mse:.4f}, Val MSE: {val_mse:.4f}")

        if show_loss:
            self.plot_losses()
    
    def plot_losses(self) -> Tuple[Figure, Axes]:

        model = self.models['horizon_0']

        train_losses = [x**2 for x in model.evals_result()['validation_0']['rmse']]
        val_losses   = [x**2 for x in model.evals_result()['validation_1']['rmse']]

        best_iter = model.get_booster().best_iteration
        n_epochs  = len(train_losses)

        epoch_patience = [False] * n_epochs
        # Mark patience epochs: those after best_iter, up to best_iter + patience
        for i in range(best_iter + 1, min(best_iter + self.patience + 1, n_epochs)):
            epoch_patience[i] = True
            
        import numpy as np

        epochs = np.arange(len(train_losses))
        patience_epochs = epochs[np.array(epoch_patience)]
        patience_train_losses = np.array(train_losses)[np.array(epoch_patience)]
        patience_val_losses = np.array(val_losses)[np.array(epoch_patience)]

        fig, axes = plt.subplots(1, 2, figsize=(16, 4))
        axes = axes.flatten()

        sns.lineplot(train_losses, color=traincolor, label='train loss', ax=axes[0])
        sns.lineplot(val_losses, color=valcolor, label='val loss', ax=axes[1])

        axes[0].scatter(patience_epochs, patience_train_losses, color='red', marker='x', label='Patience Epochs')
        axes[1].scatter(patience_epochs, patience_val_losses, color='red', marker='x', label='Patience Epochs')           

        for ax in axes:
            ax.grid()
            ax.set_ylabel('loss')
            ax.set_xlabel('epoch')
            ax.legend()

        axes[0].set_title('Training loss')      
        axes[1].set_title('Validation loss')   
        return (fig, axes)         

    def __str__(self):
        # Calculate width
        all_keys = (
            ['model name', 'model class'] +
            list(self._state.keys()) +
            list(self.config_info.get('model_hparams', {}).keys()) +
            list(self.config_info.get('global_hparams', {}).keys())
        )
        width = max(len(k) for k in all_keys) if all_keys else 20
        
        # Build output
        lines = ['<SpatioTemporalXGBModel(']
        lines.append(align('model name', self.name, width))
        lines.append(align('model class', self.model_class, width))
        lines.append('')
        
        # Status section
        status_items = {k: "✓" if v else "✗" for k, v in self._state.items()}
        lines.extend(section('status', status_items, width))
        lines.append('')
        
        # Forecasts section
        lines.extend(section('forecasts', {'forecasted': list(self.evaluation_datasets.keys())}, width))
        lines.append('')
        
        # Model hparams
        model_hparams = dict(self.config_info.get('model_hparams', {}))
        lines.extend(section('model hparams', model_hparams, width))
        lines.append('')
        
        # Global hparams
        lines.extend(section('global hparams', self.config_info.get('global_hparams', {}), width))
        
        lines.append(')>')
        
        return '\n'.join(lines)

### DeepDataLoader

In [ ]:
from src.dataloading.epidataloader import EpiDataLoader, _reorder_df
import pandas as pd
import numpy as np
import torch
from typing import Literal, Optional, Union, List, Tuple
from typing import cast
from src.dataloading.dataobjects import GraphDataLoaderEntry, GraphDataLoader

import os
import matplotlib.pyplot as plt
import copy

class DeepDataLoader(EpiDataLoader):
    """
    creates an instance with attributes
    .dataset_train
    .dataset_val
    .dataset_test

    which have an X, y, edge_index and edge_weight (the latter only when applicable!)

    X has shape: [N, F, periods] (nodes/features/periods)
    """
    def __init__(self, 
                 disease_name: str,
                 data_env_dir: str,
                 min_date:     str     = '2001-01-01',
                 max_date:     str     = '2025-01-01',
                 nuts_level:   Literal['nuts1','nuts2','nuts3'] = 'nuts3',
                 include_population: bool = False,
                 horizon_size: int     = 1,
                 horizon_leadtime:int  = 1,
                 sequence_length: int  = 1,
                 split_berlin: bool    = True):
        self.task_config = {}

        super().__init__(disease_name, data_env_dir, min_date, max_date, nuts_level, include_population, horizon_size, horizon_leadtime, sequence_length, split_berlin)
         
        self.edge_index:  Optional[torch.Tensor] = None
        self.edge_weight: Optional[torch.Tensor] = None
        self.dataloader_train, self.dataloader_val, self.dataloader_test = None, None, None

    def construct_dataloaders(self):
        """
        Creates the actual dataloaders

        Parameters:
        -----------
        periods : int
            Length of the temporal window (lookback period). Number of consecutive 
            timesteps used as input to predict the next timestep. For example:
            - periods=4 uses weeks t-1, t-2, t-3, t-4 to predict week t
            - periods=8 uses 8 weeks of history to predict the next week
            Higher values capture longer temporal dependencies but reduce training samples.

        prediction_horizon : int
            Number of timesteps ahead to predict. For example:
            - prediction_horizon=1 predicts next week (t+1)

        Attributes set:
        ---------------
        dataset_train & dataset_val & dataset_test : List[torch_Geometric.data.Data]
            each of which has the followign attributes
            - x             => [periods, node, feature]
            - y             => [node]
            - edge_index    => [2, edge_number]
            - edge_weight   => [edge_number]
        """

        X,y             = self._construct_Xy(self.data['final'])

        main_dataloader = self._construct_main_dataloader(X = X, y = y, sequence_length = self.sequence_length)
        dataloaders     = self._split_dataloader(main_dataloader = main_dataloader)
        self.dataloader_main = main_dataloader
        self.dataloader_train, self.dataloader_val, self.dataloader_test = dataloaders
        return self     
    
    def _construct_main_dataloader(self, 
                              X: torch.Tensor,
                              y: torch.Tensor, 
                              sequence_length: int) -> GraphDataLoader:
        edge_index  = self.edge_index 
        edge_weight = self.edge_weight

        if edge_index is None:
            raise ValueError('no edge index found')
        if edge_weight is None:
            raise ValueError('no edge weight found')

        dataset = []
        T       = X.shape[0]  # Total number of timesteps

        # Calculate maximum valid start position
        # Need: start + periods + prediction_horizon - 1 < T
        max_start = T - sequence_length - self.horizon_leadtime - self.horizon_size + 1
        self.max_start = max_start
        if max_start <= 0:
            raise ValueError(f"Not enough data: T={T}, periods={sequence_length}"
                            f"Need at least {sequence_length} timesteps.")

        for start in range(max_start):
            # Input window: periods consecutive timesteps
            x_seq = X[start : start + sequence_length]  # shape [periods, nodes, features]
            y_seq = y[start + sequence_length -1]
            
            data = GraphDataLoaderEntry(
                x = x_seq.clone().detach().float().permute(1, 2, 0),  # (nodes, features, periods)
                y = y_seq.clone().detach().float(),                   # (nodes, horizon)
                edge_index = edge_index,
                edge_weight =edge_weight
            )
            dataset.append(data)

        return GraphDataLoader(dataset)

    def _split_dataloader(self, 
                          main_dataloader: GraphDataLoader) -> Tuple[GraphDataLoader, GraphDataLoader, GraphDataLoader]:

        train_idx = list(self.time_splits[self.time_splits['train']].index)
        val_idx   = list(self.time_splits[self.time_splits['val']].index)
        test_idx  = list(self.time_splits[self.time_splits['test']].index)

        dataloader_train = GraphDataLoader([main_dataloader[tt] for tt in train_idx])
        dataloader_val   = GraphDataLoader([main_dataloader[tt] for tt in val_idx])
        dataloader_test  = GraphDataLoader([main_dataloader[tt] for tt in test_idx if tt < self.max_start])        
        
        return dataloader_train, dataloader_val, dataloader_test

    def _construct_Xy(self, 
                      df: pd.DataFrame) -> Tuple[torch.Tensor, torch.Tensor]:

        dfc            = df.copy()
        feature_arrays = []
        target_arrays  = []
        timestamps     = list(dfc[self.temporal_column].unique())
        time_splits    = dfc[[self.temporal_column] + self.split_columns].drop_duplicates().reset_index(drop = True)

        self.time_splits = time_splits

        for feat in self.feature_columns:
            
            # Pivot from long to wide: rows=time, columns=nodes, values=feature
            pivoted = dfc.pivot(index=['timestamp'], columns=self.id_column, values=feat).reset_index(drop = True)

            # set missing nodes to zero
            # pivoted = pivoted.reindex(index=timestamps, columns=node_ids)

            # # convert to numeric
            pivoted = pivoted.apply(pd.to_numeric, errors='coerce')
            
            # # Convert to numpy float array, replace NaNs with 0
            arr = pivoted.values
            arr = arr.astype(np.float32)             # force float32 dtype
            feature_arrays.append(arr)

        X_np = np.stack(feature_arrays, axis=-1)

        for target in self.target_horizons:

            # Pivot from long to wide: rows=time, columns=nodes, values=feature
            pivoted = dfc.pivot(index=['timestamp'], columns=self.id_column, values=target).reset_index(drop = True)

            # set missing nodes to zero
            # pivoted = pivoted.reindex(index=timestamps, columns=node_ids)

            # # convert to numeric
            pivoted = pivoted.apply(pd.to_numeric, errors='coerce')
            
            # # Convert to numpy float array, replace NaNs with 0
            arr = pivoted.values
            arr = arr.astype(np.float32)             # force float32 dtype
            target_arrays.append(arr)

        # Process target column using same approach
        y_np = np.stack(target_arrays, axis=-1)

        X = torch.tensor(X_np, dtype=torch.float)
        y = torch.tensor(y_np,dtype=torch.float)
        return X, y

    def retrieve_graph(self, 
                       graphname:     str, 
                       graphdirectory:str ='data/graphs'):
        """
        Loads graphstructure into dataloaders

        sets the following attributes in the class
        - edge_index 
        - edge_weight
        """
    
        graphpath  = os.path.join(graphdirectory, self.nuts_level, graphname)
        
        try:
            edge_index = torch.load(graphpath + '_edge_index.pt', weights_only = False)

        except Exception as e:
            raise RuntimeError(f'graph by the name of {graphname}s not found')


        # Try loading edge_weight, fallback to ones if file not found
        try:
            edge_weight = torch.load(graphpath + '_edge_weight.pt', weights_only = False)

        # else weights are uniformly 1
        except FileNotFoundError:
            num_edges   = edge_index.shape[1]
            edge_weight = torch.ones(num_edges, dtype=torch.float)

        self.edge_index  = edge_index
        self.edge_weight = edge_weight

        # self.task_config['graph'] = {'graphname': graphname,
        #                              'graphdirectory' : graphdirectory}
        return self
   
    def preview_dataloader(self, 
                        node_idx: int, 
                        timepoint: int = 22, 
                        dataset: Literal['train','val','test'] = 'test') -> 'DeepDataLoader':

        if dataset == 'train':
            df = self.dataloader_train
        elif dataset == 'val':
            df = self.dataloader_val 
        elif dataset == 'test':
            df = self.dataloader_test
        else:
            raise ValueError(f'{dataset} is an invalid dataset')
        
        if df is None:
            raise ValueError(f'no dataloader found under {dataset}')
        
        assert all(entry.x is not None for entry in df), "Some Data objects are missing 'x'"


        lags = self.lags

        lag_cols = []
        lag_cols_idx = []
        for idx, cc in enumerate(self.feature_columns):
            if 'lag' in cc:
                lag_cols.append(cc)
                lag_cols_idx.append(idx)

        if self.sequence_length > 1:
            
            dataX         = torch.stack([entry.x[node_idx, lag_cols_idx, :] for entry in df]).cpu().numpy()  # all inputs [tt, features, periods]
            last_elements = dataX[:, 0, 1:]
            to_append     = last_elements[:, ::-1]
            dataX         = np.concatenate((to_append,dataX[:,:,0]), axis = 1)


        else:
            dataX = torch.stack([entry.x[node_idx, lag_cols_idx, 0] for entry in df]).cpu().numpy()  # all inputs [tt, features, periods]


        dataY = torch.stack([entry.y[node_idx, :] for entry in df]).cpu().numpy()                # all targets of the pred horizon [tt, horizon]

        input = dataX[timepoint,:]

        input = input[::-1]

        target= dataY[timepoint]

        fig, ax = plt.subplots(figsize = (14,6))
        ax.plot(dataY[:,0], '-o' ,markersize = 5, label=f'entire timeseries for node {node_idx}')
        ax.plot(np.arange(timepoint-len(input)-lags[0]+1-self.horizon_leadtime,timepoint-lags[0]+1-self.horizon_leadtime),input, label='input for selected point', color = "#1b9e77", marker='s', markersize=10)
        ax.plot(np.arange(timepoint,timepoint+self.horizon_size),target, marker='o', markersize=10, color='#d94e4e', label='Target last point')
        ax.set_title(f'Input vs Target of node {node_idx}')
        ax.legend()
        ax.grid()
        return self

    def copy(self, deep: bool = True) -> 'DeepDataLoader':
        """
        Create a copy of the DeepDataLoader instance.
        
        Parameters:
        -----------
        deep : bool, default True
            If True, creates a deep copy (independent copy of all data).
            If False, creates a shallow copy (references to same data objects).
            
        Returns:
        --------
        DeepDataLoader
            A copy of the current instance
        """
        # Create new instance with same initialization parameters
        new_instance = DeepDataLoader(
            disease_name       = self.disease,
            data_env_dir       = self.data_env_dir,
            min_date           = self.og_min_date if isinstance(self.og_min_date, str) else self.og_min_date.strftime('%Y-%m-%d'),
            max_date           = self.max_date if isinstance(self.max_date, str) else self.max_date.strftime('%Y-%m-%d'),
            nuts_level         = cast(Literal['nuts1', 'nuts2', 'nuts3'], self.nuts_level),
            include_population = self.include_population,
            horizon_size       = self.horizon_size,
            horizon_leadtime   = self.horizon_leadtime,
            sequence_length    = self.sequence_length,
            split_berlin       = self.split_berlin
        )
        
        
        # Copy all attributes from parent class (EpiDataLoader)
        if deep:
            new_instance.target_column = copy.deepcopy(self.target_column)

            # Deep copy data structures
            if hasattr(self, 'data') and self.data:
                new_instance.data = copy.deepcopy(self.data)
            
            if hasattr(self, 'split_berlin'):
                new_instance.split_berlin = self.split_berlin
            # Copy split information
            if hasattr(self, 'time_splits'):
                new_instance.time_splits = self.time_splits.copy()
            
            if hasattr(self, 'split_summary'):
                new_instance.split_summary = self.split_summary

            if hasattr(self, 'transform_params'):
                new_instance.transform_params = self.transform_params                
                
            # Copy column definitions
            for attr in ['feature_columns', 'split_columns','target_horizons']:
                if hasattr(self, attr):
                    setattr(new_instance, attr, copy.deepcopy(getattr(self, attr)))
            
            # Copy graph structures
            if self.edge_index is not None:
                new_instance.edge_index = self.edge_index.clone()
            if self.edge_weight is not None:
                new_instance.edge_weight = self.edge_weight.clone()
                
            # Copy dataloaders (if they exist)
            for attr in ['dataloader_main', 'dataloader_train', 'dataloader_val', 'dataloader_test']:
                if hasattr(self, attr):
                    original_loader = getattr(self, attr)
                    if original_loader:
                        # Deep copy each Data object in the dataloader
                        if isinstance(original_loader, list):
                            new_loader = []
                            for data_obj in original_loader:
                                new_data = copy.deepcopy(data_obj)
                                new_loader.append(new_data)
                            setattr(new_instance, attr, new_loader)
                        else:
                            setattr(new_instance, attr, copy.deepcopy(original_loader))
        else:
            # Shallow copy - reference same objects
            for attr_name in dir(self):
                if not attr_name.startswith('_') and not callable(getattr(self, attr_name)):
                    try:
                        setattr(new_instance, attr_name, getattr(self, attr_name))
                    except AttributeError:
                        # Skip read-only attributes
                        pass
        
        # Copy other important attributes
        for attr in ['max_start', 'lags']:
            if hasattr(self, attr):
                setattr(new_instance, attr, getattr(self, attr))
        
        return new_instance

    def randomize_edge_weights(self) -> 'DeepDataLoader':
        """
        Sanity check by giving each existing edge a random weights within the same min/max range as origional

        First retrieve a graph, then use this method.

        Optionally, use the `randomize_edges` in addition.
        """

        w_min = self.edge_weight.min().item()
        w_max = self.edge_weight.max().item()

        randomized_weights = torch.rand_like(self.edge_weight) * (w_max - w_min) + w_min 
        self.edge_weight   = randomized_weights
        return self  

    def randomize_edges(self) -> 'DeepDataLoader':

        num_nodes = int(self.edge_index.max().item()) + 1
        num_edges = self.edge_index.shape[1]

        edges_from = torch.randint(low = 0, high = num_nodes, size = (num_edges,))
        edges_to   = torch.randint(low = 0, high = num_nodes, size = (num_edges,))

        self.edge_index = torch.stack([edges_from, edges_to], dim = 0)
        return self
        

def add_horizon_shifts(df: pd.DataFrame, group_column: str, target_column: str, horizons: int):
    """
    For each node, create horizon-shifted columns for the given column.
    E.g., incidence_h1, incidence_h2, ..., incidence_hN
    
    Parameters:
    - df: pandas DataFrame with columns 'node' and the target column
    - column: str, the column to shift (default 'incidence')
    - horizons: int, number of horizons/shifts to create
    
    Returns:
    - df with new columns added
    """
    df = df.copy()

    for h in range(horizons):
        col_name = f"{target_column}_h{h}"
        # Shift backward to get future values (h steps ahead)
        df[col_name] = df.groupby(group_column)[target_column].shift(-h)
    df.drop(labels = [target_column], axis = 1, inplace = True)
    
    return df.dropna()


### DeepModel

In [ ]:
# Fix the imports at the top of your files
from typing import Optional, Dict, List, Literal, Any, Union, cast
from src.utils.textformatting import section, align
from src.models.base.basemodel import BaseModel, DeepDataLoader
from src.utils import traincolor, valcolor, testcolor
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
import torch.nn as nn
from torch_geometric.nn import GCNConv, GATConv, ChebConv, GINConv
from torch_geometric_temporal.nn.recurrent import DCRNN, TGCN, A3TGCN
import pandas as pd
import numpy as np
from typing import Optional, Tuple, cast
from src.metrics.losses import spike_weighted_mse, mse, spike_timing_weighted_mse, temporal_smoothness_loss, spike_detection_loss, spatial_consistency_loss
import seaborn as sns
from src.models.utils.weights_manager import ModelWeightsManager

from src.dataloading.dataobjects import GraphDataLoader
import copy
import torch.optim as optim
from torch.optim.optimizer import Optimizer
# from torch.optim import Optimizer

from torch.optim.lr_scheduler import _LRScheduler
from abc import ABC, abstractmethod
from matplotlib.figure import Figure 
from matplotlib.axes import Axes

from tqdm import tqdm

from src.models.utils.loss.losshandler import LossHandler

from src.models.deep.strategies.base import Strategy
from src.models.deep.strategies.standard_strategy import StandardStrategy
from src.models.deep.strategies.recurrent_strategy import RecurrentStrategy

def _check_dataloader_validity(dataloader: 'DeepDataLoader') -> Tuple[GraphDataLoader, GraphDataLoader, GraphDataLoader]:

    if dataloader.dataloader_train is None or dataloader.dataloader_val is None or dataloader.dataloader_test is None:
        raise ValueError(f'dataloader invalid. No dataloaders found for train/val/test')

    train = cast(GraphDataLoader, dataloader.dataloader_train)
    val   = cast(GraphDataLoader, dataloader.dataloader_val)
    test  = cast(GraphDataLoader, dataloader.dataloader_test)

    return train, val, test


class DeepModel(BaseModel, ABC):
    """
    Parent class for all deep models.
    Uses strategy pattern to support different training and forecasting patterns.
    Subclasses only need to set the strategy in __init__ and define set_model_hparams().
    """
    
    def __init__(self, dataloader: 'GNNDataLoader', name: Optional[str] = None):
        super().__init__(dataloader, name)
        
        self.horizon_size = dataloader.horizon_size
        self.gnn_dataloader: 'GNNDataLoader' = dataloader
        self.train_loader, self.val_loader, self.test_loader = _check_dataloader_validity(dataloader)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model: Optional[torch.nn.Module] = None 
        self.optimizer: Optional[optim.optimizer.Optimizer] = None
        self.scheduler: Optional[_LRScheduler] = None
        self.weights_manager = ModelWeightsManager()

        
        self.config_info['task'] = self.gnn_dataloader.task_config
        self.config_info['child'] = 'deepmodel'
        
        self._state = {
            'model_initialized' : False,
            'global_hparams_set': False,
            'trained'           : False,
            'forecasted'        : False,
        }
        
        # Strategy - set by subclasses in __init__
        self.strategy: Strategy = StandardStrategy()
        
        self.train_losses   = []
        self.val_losses     = []
        self.epoch_patience = []
        self.learning_rates = []
        self.evaluation_datasets = {}

    def _check_state(self, required_states: List[str]) -> None:
        """Validate that required setup steps have been completed."""
        missing = [s for s in required_states if not self._state.get(s, False)]
        if missing:
            raise ValueError(
                f"Missing required setup steps: {', '.join(missing)}. "
                f"Call the corresponding methods first."
            )

    @abstractmethod
    def set_model_hparams(self) -> Any:
        pass

    def _get_optimizer(self, optimizer_name: str, lr: float, optimizer_kwargs: Dict[str, Any]) -> Optimizer:
        """Factory method to create and return optimizer"""
        if self.model is None:
            raise ValueError('Please initiate a model')

        # pylance struggles with torch typing?
        optimizer_map = {
            'adam':    optim.Adam,     # type: ignore
            'adamw':   optim.AdamW,    # type: ignore
            'sgd':     optim.SGD,      # type: ignore
            'rmsprop': optim.RMSprop,  # type: ignore
            'adagrad': optim.Adagrad,  # type: ignore
        }
        
        if optimizer_name.lower() not in optimizer_map:
            raise ValueError(f"Optimizer '{optimizer_name}' not supported. Choose from: {list(optimizer_map.keys())}")
        
        optimizer_class = optimizer_map[optimizer_name.lower()]

        return optimizer_class(self.model.parameters(), lr=lr, **optimizer_kwargs)

    def _get_scheduler(self, scheduler_name: str, optimizer: Optimizer, scheduler_kwargs: Dict[str, Any]) -> _LRScheduler:
        """Factory method to create and return scheduler"""
        if self.model is None:
            raise ValueError('Please initiate a model')
        scheduler_map = {
            'step':        torch.optim.lr_scheduler.StepLR,
            'exponential': torch.optim.lr_scheduler.ExponentialLR,
            'cosine':      torch.optim.lr_scheduler.CosineAnnealingLR,
            'plateau':     torch.optim.lr_scheduler.ReduceLROnPlateau,
            'cyclic':      torch.optim.lr_scheduler.CyclicLR,
            'onecycle':    torch.optim.lr_scheduler.OneCycleLR,
            'multistep':   torch.optim.lr_scheduler.MultiStepLR,
            'lambda':      torch.optim.lr_scheduler.LambdaLR,
        }
        
        if scheduler_name.lower() not in scheduler_map:
            raise ValueError(f"Scheduler '{scheduler_name}' not supported. Choose from: {list(scheduler_map.keys())}")
        
        scheduler_class = scheduler_map[scheduler_name.lower()]
        return scheduler_class(optimizer, **scheduler_kwargs)

    def _set_strategy(self, strategy: Strategy) -> None:
        """Allow subclasses to specify their strategy"""
        self.strategy = strategy

    def set_global_hparams(self, 
                            lr: float = 0.001,
                            n_epochs: int = 5,
                            patience: int = 15,
                            min_delta: float = 1e-4,                            
                            optimizer: str = 'adam',
                            optimizer_kwargs: Optional[Dict[str, Any]] = None,
                            scheduler: Optional[str] = 'step',
                            scheduler_kwargs: Optional[Dict[str, Any]] = None,
                            loss: str = 'mse',
                            loss_kwargs: Optional[Dict[str, Any]] = None                            
                            ):
        """Prepares model for training using global hyperparameters."""
        self._check_state(['model_initialized'])

        global_params_config = {
            'lr': lr,
            'n_epochs': n_epochs,
            'patience': patience,
            'min_delta': min_delta,                       
            'optimizer': optimizer,
            'optimizer_kwargs': optimizer_kwargs,
            'scheduler': scheduler,
            'scheduler_kwargs': scheduler_kwargs,
            'loss': loss,
            'loss_kwargs' : loss_kwargs
        }
        
        self.global_hparams_set = True
        self.n_epochs  = n_epochs
        self.patience  = patience
        self.min_delta = min_delta
        self.loss = LossHandler(loss, loss_kwargs=loss_kwargs)  

        if optimizer_kwargs is None:
            optimizer_kwargs = {}
        
        if scheduler_kwargs is None:
            default_scheduler_kwargs = {
                'step':        {'step_size': 15, 'gamma': 0.8},
                'exponential': {'gamma': 0.95},
                'plateau':     {'mode': 'min', 'factor': 0.5, 'patience': 10, 'verbose': True}
            }
            scheduler_kwargs = default_scheduler_kwargs.get(scheduler, {}) if scheduler else {}

        self.optimizer = self._get_optimizer(optimizer, lr, optimizer_kwargs)
        
        if scheduler:
            self.scheduler = self._get_scheduler(scheduler, self.optimizer, scheduler_kwargs)
        else:
            self.scheduler = None

        self.config_info['global_hparams'] = global_params_config
        self._state['global_hparams_set'] = True
        return self

    def train(self,
              verbose: Literal[0,1,2] = 1,
              dataloader_snapshot: bool = True,
              show_loss: bool = True):
        """
        Unified training loop that works for both standard and recurrent models.
        The strategy handles all the differences.
        """
        if self.model is None:
            raise ValueError('Please initiate a model')
        
        if self.optimizer is None:
            raise ValueError('no valid optimizer found')

        if self.scheduler is None:
            raise ValueError('no valid scheduler found')
        
        self._check_state(['model_initialized', 'global_hparams_set'])

        if dataloader_snapshot:
            print(f'Dataloader Snapshot: {self.train_loader[0]}')

        if verbose == 1:
            verbose_loops = list(np.arange(1, self.n_epochs + 1, step=10))
        elif verbose == 2:
            verbose_loops = list(np.arange(1, self.n_epochs + 1))
        else:
            verbose_loops = []

        self.model.train()
        best_val_loss = float('inf')
        patience_counter = 0
        best_model_state = None

        list_val_loss = []
        list_train_loss = []
        list_patience = []

        L_train = len(list(self.train_loader))
        L_val = len(list(self.val_loader))
        
        if verbose == 0:
            epoch_iter = tqdm(range(self.n_epochs), desc="Training epochs")
        else:
            epoch_iter = range(self.n_epochs)

        for epoch in epoch_iter:
            # Reset state at epoch start
            self.strategy.reset_state()

            # ======================== TRAINING PHASE ========================
            total_loss = 0
            
            for snapshot in self.train_loader:
                snapshot = snapshot.to(self.device)
                loss_val = self.strategy.training_step(
                    self.model, snapshot, self.optimizer, self.loss
                )
                total_loss += loss_val
            
            train_mse = total_loss / L_train
            list_train_loss.append(train_mse)

            # ======================== VALIDATION PHASE ========================
            self.model.eval()
            val_loss = 0
            
            # Reset state for validation
            self.strategy.reset_state()

            with torch.no_grad():
                for snapshot in self.val_loader:
                    snapshot = snapshot.to(self.device)
                    loss_val = self.strategy.validation_step(
                        self.model, snapshot, self.loss
                    )
                    val_loss += loss_val
            
            val_mse = val_loss / L_val
            list_val_loss.append(val_mse)
            
            current_lr = self.optimizer.param_groups[0]['lr']
            self.learning_rates.append(current_lr)
            
            self.model.train()
            
            # Check if validation loss improved
            val_improved = val_mse < (best_val_loss - self.min_delta)

            if val_improved:
                best_val_loss = val_mse
                patience_counter = 0
                best_model_state = self.model.state_dict().copy()
                if epoch in verbose_loops:
                    print(f"Epoch {epoch} train loss: {train_mse:.4f}, val loss: {val_mse:.4f} ✓ (new best)")
                list_patience.append(False)

            else:
                patience_counter += 1
                if epoch in verbose_loops:
                    print(f"Epoch {epoch} train loss: {train_mse:.4f}, val loss: {val_mse:.4f} (patience: {patience_counter}/{self.patience})")
                list_patience.append(True)
                
                if patience_counter >= self.patience:
                    print(f"Early stopping: Validation loss hasn't improved for {self.patience} epochs")
                    if best_model_state is not None:
                        self.model.load_state_dict(best_model_state)
                        print(f"Restored model from best validation loss: {best_val_loss:.4f}")
                    break
            
            # Step scheduler
            if isinstance(self.scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                self.scheduler.step(val_mse)
            else:
                self.scheduler.step()
            
            self.train_losses = list_train_loss
            self.val_losses = list_val_loss
            self.epoch_patience = list_patience

        self._state['trained'] = True
        if show_loss:
            self.plot_losses()

    def forecast(self, dataset: Literal['train','val','test'] = 'test'):
        """
        Unified forecasting loop that works for both standard and recurrent models.
        The strategy handles hidden state management.
        """
        self._check_state(['model_initialized', 'global_hparams_set', 'trained'])
        if self.model is None:
            raise ValueError('Please initiate a model')

        self.model.eval()
        
        predictions = []
        labels = []

        eval_df = self.gnn_dataloader.data['final'][self.gnn_dataloader.data['final'][dataset]]

        if dataset == 'train':
            dataloader = self.train_loader
        elif dataset == 'val':
            dataloader = self.val_loader
        elif dataset == 'test':
            dataloader = self.test_loader
        else:
            raise ValueError(f'dataset must be either "train", "val" or "test"')

        # Reset state before forecasting
        self.strategy.reset_state()
        
        loss = 0
        with torch.no_grad():
            for snapshot in tqdm(dataloader, desc=f"Forecasting {dataset}"):
                snapshot = snapshot.to(self.device)
                y_hat, loss_val = self.strategy.forecast_step(
                    self.model, snapshot, self.loss
                )
                loss += loss_val
                labels.append(snapshot.y)
                predictions.append(y_hat)

        loss = loss / len(dataloader)
        print(f"{dataset.capitalize()} loss: {loss:.4f}")
        setattr(self, f'{dataset}_loss', loss)

        # ======================== FORMAT PREDICTIONS ========================
        tensor_list_cpu = [t.detach().cpu() for t in predictions]
        stacked = torch.stack(tensor_list_cpu)   
        num_timepoints, n_nodes, horizon = stacked.shape
        reshaped = stacked.view(num_timepoints * n_nodes, horizon).numpy()  

        timepoints = np.repeat(np.arange(num_timepoints), n_nodes)
        nodes = np.tile(np.arange(n_nodes), num_timepoints)
        index = pd.MultiIndex.from_arrays([timepoints, nodes], names=['timestamp_idx', 'node'])

        columns = [f"pred_h{h}" for h in range(horizon)]
        df_pred = pd.DataFrame(reshaped, index=index, columns=columns).reset_index(drop=False)

        # ======================== FORMAT TARGETS ========================
        tensor_list_cpu = [t.detach().cpu() for t in labels]
        stacked = torch.stack(tensor_list_cpu)   
        num_timepoints, n_nodes, horizon = stacked.shape
        reshaped = stacked.view(num_timepoints * n_nodes, horizon).numpy()  

        timepoints = np.repeat(np.arange(num_timepoints), n_nodes)
        nodes = np.tile(np.arange(n_nodes), num_timepoints)
        index = pd.MultiIndex.from_arrays([timepoints, nodes], names=['timestamp_idx', 'node'])

        columns = self.gnn_dataloader.target_horizons
        df_target = pd.DataFrame(reshaped, index=index, columns=columns).reset_index(drop=False)
        
        merged = pd.merge(df_pred, df_target, on=['timestamp_idx','node'])
        merged['timestamp_idx'] = merged['timestamp_idx'] + (self.gnn_dataloader.sequence_length - 1)
        timestamp_map = eval_df[['timestamp']].drop_duplicates().reset_index(drop=True).reset_index(drop=False).rename(columns={'index': 'timestamp_idx'})
        merged['timestamp'] = merged['timestamp_idx'].map(dict(zip(timestamp_map['timestamp_idx'], timestamp_map['timestamp'])))
        
        formatted_eval = pd.merge(merged[['timestamp', 'node'] + [f'pred_h{hh}' for hh in range(horizon)]], eval_df, on=['timestamp','node'], how='right')

        columns_context = [self.gnn_dataloader.temporal_column, self.gnn_dataloader.id_column] + self.gnn_dataloader.feature_columns + self.gnn_dataloader.split_columns + [self.gnn_dataloader.target_horizons[0]]

        horizon_prediction_dict = {}

        for hh in range(horizon):
            horizon_predictions = formatted_eval[columns_context + [f'pred_h{hh}']]
            horizon_predictions = horizon_predictions.rename(columns={
                f'pred_h{hh}': 'pred',
                f'{self.gnn_dataloader.target_horizons[0]}': f'{self.gnn_dataloader.target_column}'
            })
            horizon_predictions['pred'] = horizon_predictions['pred'].shift(-hh)

            horizon_prediction_dict['transformed'] = {f'horizon_{hh}': horizon_predictions}
            horizon_prediction_dict['nontransformed'] = {f'horizon_{hh}': self._denorm_predictions(horizon_predictions)}

        self.evaluation_datasets[dataset] = horizon_prediction_dict
        self._state['forecasted'] = True
        return self

    def plot_losses(self) -> Tuple[Figure, Axes]:
        """Returns plot of train and val losses per epoch"""
        if self.model is None:
            raise ValueError('Please initiate a model')

        epochs = np.arange(len(self.train_losses))
        patience_epochs = epochs[np.array(self.epoch_patience)]
        patience_train_losses = np.array(self.train_losses)[np.array(self.epoch_patience)]
        patience_val_losses = np.array(self.val_losses)[np.array(self.epoch_patience)]
        patience_learningrates = np.array(self.learning_rates)[np.array(self.epoch_patience)]

        fig, axes = plt.subplots(1, 3, figsize=(24, 4))
        axes = axes.flatten()

        sns.lineplot(self.train_losses, color=traincolor, label='train loss', ax=axes[0])
        sns.lineplot(self.val_losses, color=valcolor, label='val loss', ax=axes[1])
        sns.lineplot(self.learning_rates, color='green', label='learning rate', ax=axes[2])
        
        axes[0].scatter(patience_epochs, patience_train_losses, color='red', marker='x', label='Patience Epochs')
        axes[1].scatter(patience_epochs, patience_val_losses, color='red', marker='x', label='Patience Epochs')   
        axes[2].scatter(patience_epochs, patience_learningrates, color='red', marker='x', label='Patience Epochs')           

        for ax in axes:
            ax.grid()
            ax.set_ylabel('loss')
            ax.set_xlabel('epoch')
            ax.legend()

        axes[0].set_title('Training loss')      
        axes[1].set_title('Validation loss')    
        axes[2].set_title('Learning Rate Schedule')
        axes[2].set_ylabel('Learning Rate')
        axes[2].set_yscale('log')

        return (fig, axes)
    
    def run_snapshot(self, index: int = 0, debug: bool = False):
        """
        Run a single snapshot through the model and print output for validation/debugging.
        """
        if self.model is None:
            raise ValueError('Please initiate a model')

        self.model.eval()  # Set model to eval mode

        # Get snapshot from dataloader
        snapshot = self.train_loader[index]
        snapshot = snapshot.to(self.device)

        with torch.no_grad():
            # Forward pass
            y_hat = self.model(snapshot.x, snapshot.edge_index, snapshot.edge_weight, debug=debug)

        # Ground truth
        y_true = snapshot.y.to(self.device)

        # Compute error
        loss = F.mse_loss(y_hat, y_true).item()

        # Print summary
        if debug:
            print(f"\n🧪 Snapshot {index} validation summary:")
            print(f"➡️  Predicted shape: {y_hat.shape}")
            print(f"➡️  Ground truth shape: {y_true.shape}")

        print(f"✅ Snapshot ran successfully")

        return y_hat, y_true

    def save_weights(self,
                     filename: Optional[str] = None,
                     save_optimizer: bool = True,
                     save_scheduler: bool = True,
                     metadata: Optional[Dict] = None) -> str:
        """
        Save model weights only (not configuration).
        
        For saving configuration, use the parent's save_model() method.
        
        Parameters:
        ----------
        filename : Optional[str]
            Custom filename (without extension)
        save_optimizer : bool
            Save optimizer state for training resumption
        save_scheduler : bool
            Save scheduler state for training resumption
        metadata : Optional[Dict]
            Additional metadata to store
            
        Returns:
        -------
        str : Path to saved weights file
        
        Example:
        -------
        >>> # Save config once (in BaseModel)
        >>> model.save_model()  # Saves config to YAML
        >>> 
        >>> # Save weights multiple times during training
        >>> model.save_weights(filename='epoch_50')
        >>> model.save_weights(filename='epoch_100')
        >>> model.save_weights(filename='best_model', save_optimizer=False)
        """
        if self.model is None:
            raise ValueError('No model to save')
        
        weights_path = self.weights_manager.save_weights(
            model=self,
            filename=filename,
            save_optimizer=save_optimizer,
            save_scheduler=save_scheduler,
            metadata=metadata
        )
        
        return weights_path
    
    def _load_weights(self,
                     model_number: int) -> 'DeepModel':
        """
        Load model weights.
        Model architecture should already be initialized via set_model_hparams().
        
        Parameters:
        ----------
        weights_path : str
            Path to the weights file
        load_optimizer : bool
            Load optimizer state
        load_scheduler : bool
            Load scheduler state
        strict : bool
            Strictly enforce key matching
            
        Returns:
        -------
        self : For method chaining
        """
        metadata = self.weights_manager.load_weights(
            model=self,
            model_number=model_number
        )
        
        # Update config_info with loaded metadata
        if metadata.get('config_id'):
            self.config_info['id'] = metadata['config_id']
        
        return self
    
    def load_config(self, model_name: str):

        cfg      = self.config_manager.load_entry(entry_name = model_name)
        entry_id = cfg['id']

        if self.__class__.__name__.lower() != cfg['model']:
            raise ValueError(f'The config loaded is one of a {cfg["model"]} which does not work for {self.name}, given it is a {self.__class__.__name__}')

        self.set_model_hparams(**cfg['model_hparams'])
        self.set_global_hparams(**cfg['global_hparams'])
        self._load_weights(model_number= entry_id)
        print(f"✓ Model loaded")

    def __str__(self):
        # Calculate width
        all_keys = (
            ['model name', 'model class'] +
            list(self._state.keys()) +
            list(self.config_info.get('model_hparams', {}).keys()) +
            list(self.config_info.get('global_hparams', {}).keys())
        )
        width = max(len(k) for k in all_keys) if all_keys else 20
        
        # Build output
        lines = ['<DeepModel(']
        lines.append(align('model name', self.name, width))
        lines.append(align('model class', self.model_class, width))
        lines.append('')
        
        # Status section
        status_items = {k: "✓" if v else "✗" for k, v in self._state.items()}
        lines.extend(section('status', status_items, width))
        lines.append('')
        
        # Forecasts section
        lines.extend(section('forecasts', {'forecasted': list(self.evaluation_datasets.keys())}, width))
        lines.append('')
        
        # Model hparams
        model_hparams = dict(self.config_info.get('model_hparams', {}))
        model_hparams['strategy'] = self.strategy
        lines.extend(section('model hparams', model_hparams, width))
        lines.append('')
        
        # Global hparams
        lines.extend(section('global hparams', self.config_info.get('global_hparams', {}), width))
        
        lines.append(')>')
        
        return '\n'.join(lines)

# New stuff

In [ ]:
from src.dataloading.epidataloader import EpiDataLoader
import torch
import os
from src.dataloading.dataobjects import GraphDataLoaderEntry, GraphDataLoader
from src.utils import get_data_env

class ShallowDataLoader(EpiDataLoader):
    """
    ...
    """
    def __init__(self, 
                 disease_name: str,
                 data_env_dir: str,
                 min_date:     str     = '2001-01-01',
                 max_date:     str     = '2025-01-01',
                 nuts_level:   Literal['nuts1','nuts2','nuts3'] = 'nuts3',
                 include_population: bool = False,
                 horizon_size: int     = 1,
                 horizon_leadtime:int  = 1,
                 sequence_length: int  = 1,
                 split_berlin: bool    = True):
        self.task_config = {}

        super().__init__(disease_name, data_env_dir, min_date, max_date, nuts_level, include_population, horizon_size, horizon_leadtime, sequence_length, split_berlin)
         
        self.edge_index:  Optional[torch.Tensor] = None
        self.edge_weight: Optional[torch.Tensor] = None
        self.dataloader_train, self.dataloader_val, self.dataloader_test = None, None, None    

    def retrieve_graph(self, 
                       graphname:     str, 
                       graphdirectory:str ='data/graphs'):
        """
        Loads graphstructure into dataloader by importing the edge_indices
        associated with the graphname into self.edge_index.

        Note that edge weights are not used

        self.neighbor_dict is set

        See Also:
        --------
        _build_neighbor_dict()
        """
    
        graphpath  = os.path.join(graphdirectory, self.nuts_level, graphname)
        
        try:
            edge_index = torch.load(graphpath + '_edge_index.pt', weights_only = False)

        except Exception as e:
            raise RuntimeError(f'graph by the name of {graphname}s not found')

        self.edge_index  = edge_index
        self.neighbor_dict  = self._build_neighbor_dict()
        return self
   
    def construct_dataloaders(self):
        """

        """

        self.sequential_data   = self._make_sequential_data()


        # X,y             = self._construct_Xy(self.data['final'])

        # main_dataloader = self._construct_main_dataloader(X = X, y = y, sequence_length = self.sequence_length)
        # dataloaders     = self._split_dataloader(main_dataloader = main_dataloader)

        # self.dataloader_main = main_dataloader
        # self.dataloader_train, self.dataloader_val, self.dataloader_test = dataloaders
        return self     
    
    def _construct_main_dataloader(self, 
                              X: torch.Tensor,
                              y: torch.Tensor, 
                              sequence_length: int) -> GraphDataLoader:
        edge_index  = self.edge_index 
        edge_weight = self.edge_weight

        if edge_index is None:
            raise ValueError('no edge index found')
        if edge_weight is None:
            raise ValueError('no edge weight found')

        dataset = []
        T       = X.shape[0]  # Total number of timesteps

        # Calculate maximum valid start position
        # Need: start + periods + prediction_horizon - 1 < T
        max_start = T - sequence_length - self.horizon_leadtime - self.horizon_size + 1
        self.max_start = max_start
        if max_start <= 0:
            raise ValueError(f"Not enough data: T={T}, periods={sequence_length}"
                            f"Need at least {sequence_length} timesteps.")

        for start in range(max_start):
            # Input window: periods consecutive timesteps
            x_seq = X[start : start + sequence_length]  # shape [periods, nodes, features]
            y_seq = y[start + sequence_length -1]
            
            data = GraphDataLoaderEntry(
                x = x_seq.clone().detach().float().permute(1, 2, 0),  # (nodes, features, periods)
                y = y_seq.clone().detach().float(),                   # (nodes, horizon)
                edge_index = edge_index,
                edge_weight =edge_weight
            )
            dataset.append(data)

        return GraphDataLoader(dataset)
    
    def _split_dataloader(self, 
                          main_dataloader: GraphDataLoader) -> Tuple[GraphDataLoader, GraphDataLoader, GraphDataLoader]:

        train_idx = list(self.time_splits[self.time_splits['train']].index)
        val_idx   = list(self.time_splits[self.time_splits['val']].index)
        test_idx  = list(self.time_splits[self.time_splits['test']].index)

        dataloader_train = GraphDataLoader([main_dataloader[tt] for tt in train_idx])
        dataloader_val   = GraphDataLoader([main_dataloader[tt] for tt in val_idx])
        dataloader_test  = GraphDataLoader([main_dataloader[tt] for tt in test_idx if tt < self.max_start])        
        
        return dataloader_train, dataloader_val, dataloader_test

    def _construct_Xy(self, 
                      df: pd.DataFrame) -> Tuple[torch.Tensor, torch.Tensor]:
        """ 
        extracts and returns X, y from final epidata data
        """

        dfc            = df.copy()
        feature_arrays = []
        target_arrays  = []
        timestamps     = list(dfc[self.temporal_column].unique())
        time_splits    = dfc[[self.temporal_column] + self.split_columns].drop_duplicates().reset_index(drop = True)

        self.time_splits = time_splits

        for feat in self.feature_columns:
            
            # Pivot from long to wide: rows=time, columns=nodes, values=feature
            pivoted = dfc.pivot(index=['timestamp'], columns=self.id_column, values=feat).reset_index(drop = True)

            # set missing nodes to zero
            # pivoted = pivoted.reindex(index=timestamps, columns=node_ids)

            # # convert to numeric
            pivoted = pivoted.apply(pd.to_numeric, errors='coerce')
            
            # # Convert to numpy float array, replace NaNs with 0
            arr = pivoted.values
            arr = arr.astype(np.float32)             # force float32 dtype
            feature_arrays.append(arr)

        X_np = np.stack(feature_arrays, axis=-1)

        for target in self.target_horizons:

            # Pivot from long to wide: rows=time, columns=nodes, values=feature
            pivoted = dfc.pivot(index=['timestamp'], columns=self.id_column, values=target).reset_index(drop = True)

            # set missing nodes to zero
            # pivoted = pivoted.reindex(index=timestamps, columns=node_ids)

            # # convert to numeric
            pivoted = pivoted.apply(pd.to_numeric, errors='coerce')
            
            # # Convert to numpy float array, replace NaNs with 0
            arr = pivoted.values
            arr = arr.astype(np.float32)             # force float32 dtype
            target_arrays.append(arr)

        # Process target column using same approach
        y_np = np.stack(target_arrays, axis=-1)

        X = torch.tensor(X_np, dtype=torch.float)
        y = torch.tensor(y_np,dtype=torch.float)
        return X, y
    
    def _build_neighbor_dict(self) -> Dict[int, List[int]]:
        """
        Build dictionary mapping each node to its neighbors.
        """
        
        neighbor_dict   = {}
        edge_list       = self.edge_index.t().tolist()
    
        for src, dst in edge_list:
            if src not in neighbor_dict:
                neighbor_dict[src] = set()
            neighbor_dict[src].add(dst)
        
        neighbor_dictionary = {k: sorted(list(v)) for k, v in neighbor_dict.items()}
        return neighbor_dictionary
    
    def _unpack_dataloader(self, loader):
        """
        Extract data from GNN dataloader format into flat dataframe (vectorized),
        and augment with neighbor-aggregated features.
        """

        num_nodes, num_features, sequence_length = loader[0].x.shape
        num_snapshots = len(loader)

        features = np.empty((num_snapshots, num_nodes, num_features, sequence_length), dtype=np.float32)
        targets = np.empty((num_snapshots, num_nodes, self.horizon_size), dtype=np.float32)

        for t_idx, snapshot in enumerate(loader):
            features[t_idx] = snapshot.x.numpy()
            targets[t_idx] = snapshot.y.numpy()

        # Reverse time axis to match naming convention (t, t-1, ...)
        features = features[..., ::-1]

        # Create feature column names
        feature_cols = []
        for feat_name in self.dataloader.feature_columns:
            for t in range(sequence_length):
                suffix = "_t" if t == 0 else f"_t-{t}"
                feature_cols.append(f"{feat_name}{suffix}")

        # Flatten arrays
        features_flat = features.reshape(num_snapshots * num_nodes, -1)
        targets_flat = targets.reshape(num_snapshots * num_nodes, self.horizon_size)

        # Create base dataframe
        t_idx_arr = np.repeat(np.arange(num_snapshots), num_nodes)
        node_arr = np.tile(np.arange(num_nodes), num_snapshots)

        df = pd.DataFrame(features_flat, columns=feature_cols)
        df['t_idx'] = t_idx_arr
        df['node'] = node_arr

        # Add target columns
        for h in range(self.horizon_size):
            df[f"target_h{h}"] = targets_flat[:, h]

        # === Vectorized Neighbor Aggregation ===
        # 1. Create neighbor mapping table — only valid neighbors
        rows = []
        for node, neighbors in self.neighbor_dict.items():
            for nbr in neighbors:
                if nbr != node:  # Optional: skip self if needed
                    rows.append({'node': node, 'neighbor': nbr})
        neighbor_map = pd.DataFrame(rows)  # [node, neighbor]

        # ⛔️ If the above DataFrame is empty (e.g. only self-loops), stop early
        if neighbor_map.empty:
            print("⚠️ No neighbors found — skipping aggregation.")
            return df

        # 2. Get node-level features only (no targets, no duplicate columns)
        base_feats = df.drop(columns=[c for c in df.columns if c.startswith('target')])

        # 3. Merge to assign neighbors
        df_neighbors = neighbor_map.merge(base_feats, left_on='neighbor', right_on='node')
        df_neighbors = df_neighbors.rename(columns={'node_x': 'node', 't_idx': 't_idx', 'node_y': 'neighbor'})

        # 4. Aggregate neighbor features
        agg_feature_cols = [col for col in base_feats.columns if col not in ['node', 't_idx']]

        neighbor_agg = (
            df_neighbors
            .groupby(['node', 't_idx'])[agg_feature_cols]
            .mean()
            .add_suffix('_neighbor')
            .reset_index()
        )

        # 5. Merge neighbor features back into original
        df_final = df.merge(neighbor_agg, on=['node', 't_idx'], how='left')

        return df_final
    
    ###############

    def _make_sequential_data(self) -> pd.DataFrame:

        """ 
        return df with sequential data in columns
        """

        df              = self.data['final']
        feature_columns = self.feature_columns
        sequence_length = self.sequence_length

        df = df.copy()
        df.sort_values(['node', 'timestamp'], inplace=True)

        lagged_frames = []  # List of DataFrames to concatenate later

        for lag in range(1, sequence_length + 1):
            shifted = df.groupby('node')[feature_columns].shift(lag)
            shifted.columns = [f"{col}_t-{lag}" for col in feature_columns]
            lagged_frames.append(shifted)
        

        # Concatenate lagged features horizontally (axis=1)
        lagged_df = pd.concat(lagged_frames, axis=1)
        lagged_feature_columns = [
        f"{col}_t-{lag}" 
        for lag in range(1, sequence_length + 1)
        for col in feature_columns
    ]
        self.feature_columns.append(lagged_feature_columns)
        # Concatenate with original DataFrame only once
        df_final = pd.concat([df, lagged_df], axis=1)

        return df_final.dropna().reset_index(drop = True)




In [54]:
disease_name    = 'influenza'
nuts_level      = 'nuts3'
min_date        ='2006-05-15'
max_date        = '2020-06-01'
split_trainval  = '2018-06-01'
split_valtest   = '2019-06-01'
split_berlin    = False


horizon_size    = 1
horizon_leadtime= 3
sequence_length = 12
lags            = 1

xgboost_loader_basis = ShallowDataLoader(disease_name, get_data_env(), nuts_level=nuts_level, min_date=min_date,max_date=max_date, include_population=False, horizon_size = horizon_size, horizon_leadtime = horizon_leadtime, sequence_length=sequence_length, split_berlin=split_berlin)
xgboost_loader_basis.add_time_features()
xgboost_loader_basis.log_transform_target()
xgboost_loader_basis.set_splits(split_trainval, split_valtest)
xgboost_loader_basis.normalize()
xgboost_loader_basis.add_lagged_features(lags = lags)
xgboost_loader_basis.finalize()
xgboost_loader_basis.retrieve_graph('boolean_neighbors_selfmean')
xgboost_loader_basis.construct_dataloaders()

Dataloader temporal windowing: extending data collection from 2006-05-15 to 2006-01-30 (+15 weeks)
berlin districts removed


EpiDataLoader(disease=influenza, nuts_level=nuts3, min_date=2006-01-30, max_date=2020-06-01, horizon_size=1, horizon_leadtime=3, sequence_length=12)

In [55]:
xgboost_loader_basis.neighbor_dict

{247: [247, 255],
 86: [70, 72, 86, 87, 88, 109, 111, 112, 144],
 32: [26, 28, 31, 32, 101, 102],
 258: [258, 264],
 89: [65, 69, 77, 89, 94],
 229: [229, 231, 237, 238, 241],
 269: [269, 273],
 329: [324, 329, 333, 334, 342],
 377: [331, 338, 341, 363, 364, 368, 377],
 121: [116, 120, 121, 122, 126, 127, 295, 300],
 19: [16, 18, 19, 21, 23, 26, 33, 42, 367, 369],
 200: [179, 193, 194, 199, 200, 201, 202, 214],
 5: [2, 5, 14, 15, 35, 37, 348, 350],
 52: [51, 52, 54, 55, 57, 95],
 332: [324, 325, 328, 332, 334, 337, 338, 372, 376],
 324: [324, 328, 329, 330, 332, 333, 334, 336, 338, 341],
 322: [167, 172, 173, 177, 318, 320, 322],
 74: [63, 64, 65, 68, 71, 72, 74, 75, 79, 80, 87, 109],
 114: [96, 98, 103, 107, 110, 111, 114, 115],
 165: [165, 175, 193, 198],
 193: [165, 171, 175, 185, 192, 193, 194, 198, 200, 201],
 219: [210, 219, 220, 221, 313],
 373: [371, 373, 374, 375, 384, 387],
 177: [160, 164, 167, 169, 172, 174, 177, 322],
 336: [324, 327, 330, 333, 336, 340],
 101: [28, 30, 32

In [60]:
neighbor_agg = compute_neighbor_aggregates(xgboost_loader_basis.sequential_data, xgboost_loader_basis.neighbor_dict, xgboost_loader_basis.feature_columns)

In [62]:
xgboost_loader_basis.feature_columns

['timestamp_sin',
 'timestamp_cos',
 'incidence_lag3',
 ['timestamp_sin_t-1',
  'timestamp_cos_t-1',
  'incidence_lag3_t-1',
  'timestamp_sin_t-2',
  'timestamp_cos_t-2',
  'incidence_lag3_t-2',
  'timestamp_sin_t-3',
  'timestamp_cos_t-3',
  'incidence_lag3_t-3',
  'timestamp_sin_t-4',
  'timestamp_cos_t-4',
  'incidence_lag3_t-4',
  'timestamp_sin_t-5',
  'timestamp_cos_t-5',
  'incidence_lag3_t-5',
  'timestamp_sin_t-6',
  'timestamp_cos_t-6',
  'incidence_lag3_t-6',
  'timestamp_sin_t-7',
  'timestamp_cos_t-7',
  'incidence_lag3_t-7',
  'timestamp_sin_t-8',
  'timestamp_cos_t-8',
  'incidence_lag3_t-8',
  'timestamp_sin_t-9',
  'timestamp_cos_t-9',
  'incidence_lag3_t-9',
  'timestamp_sin_t-10',
  'timestamp_cos_t-10',
  'incidence_lag3_t-10',
  'timestamp_sin_t-11',
  'timestamp_cos_t-11',
  'incidence_lag3_t-11',
  'timestamp_sin_t-12',
  'timestamp_cos_t-12',
  'incidence_lag3_t-12']]

In [57]:
def compute_neighbor_aggregates(df: pd.DataFrame, neighbor_dict: dict, feature_columns: list[str]):
    df = df.copy()
    df.sort_values(['node', 'timestamp'], inplace=True)
    
    # For fast access
    df.set_index(['node', 'timestamp'], inplace=True)
    
    rows = []
    for node in neighbor_dict:
        neighbors = neighbor_dict[node]
        for timestamp in df.loc[node].index:
            # For each node-timestamp, get neighbor features
            try:
                neighbor_feats = df.loc[(neighbors, timestamp)][feature_columns]
                row = {
                    'node': node,
                    'timestamp': timestamp,
                }
                for col in feature_columns:
                    row[f'{col}_neighbor_mean'] = neighbor_feats[col].mean()
                    row[f'{col}_neighbor_max'] = neighbor_feats[col].max()
                rows.append(row)
            except KeyError:
                continue  # Missing data, skip
    return pd.DataFrame(rows)